In [1]:
import pandas as pd

# Load dataset
data = pd.read_csv(r'F:\Shweta MSDA Sem2\Data 245 Machine Learning\waterdata\combined_water_quality_pipeline_data.csv')

# Display the first few rows of the dataset for verification
print(data.head())


   Index        pH      Iron   Nitrate    Chloride           Lead      Zinc  \
0      0  8.332988  0.000083  8.605777  122.799772   3.710000e-52  3.434827   
1      1  6.917863  0.000081  3.734167  227.029851   7.850000e-94  1.245317   
2      2  5.443762  0.020106  3.816994  230.995630   5.290000e-76  0.528280   
3      3  7.955339  0.143988  8.224944  178.129940  4.000000e-176  4.027879   
4      4  8.091909  0.002167  9.925788  186.540872  4.170000e-132  3.807511   

            Color  Turbidity  Fluoride  ...  Source  Water Temperature  \
0       Colorless   0.022683  0.607283  ...     NaN                NaN   
1    Faint Yellow   0.019007  0.622874  ...    Lake          15.348981   
2    Light Yellow   0.319956  0.423423  ...   River          11.643467   
3  Near Colorless   0.166319  0.208454  ...  Ground          10.092392   
4    Light Yellow   0.004867  0.222912  ...  Spring          15.249416   

   Air Temperature     Month   Day  Time of Day  Target           City  \
0     

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.feature_selection import RFE
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import resample

In [3]:
categorical_cols = ['City', 'Pipeline_Material', 'Color', 'Odor', 'Source', 'Month', 'Time of Day']
for col in categorical_cols:
    data[col] = data[col].astype(str)
    data[col] = LabelEncoder().fit_transform(data[col])

data.replace([np.inf, -np.inf], np.nan, inplace=True)
data = pd.DataFrame(SimpleImputer(strategy='mean').fit_transform(data), columns=data.columns)

In [4]:
# === Step 2. 
X = data.drop(columns=['Target'])
y = data['Target'].astype(int)
X_columns = X.columns
X_df = pd.DataFrame(X, columns=X_columns)

# === Step 3. 
X_train_df, X_test_df, y_train, y_test = train_test_split(X_df, y, test_size=0.2, random_state=42)

# === Step 4.
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_df)
X_test_scaled = scaler.transform(X_test_df)

In [8]:
# === Step 5
rfe_estimator = LinearSVC(random_state=42, max_iter=2000)
rfe = RFE(estimator=rfe_estimator, n_features_to_select=10)
rfe.fit(X_train_scaled, y_train)

# === Step 6.
rfe_features = X_columns[rfe.support_].tolist()
important_features = [
    'Odor', 'Chloride', 'Manganese', 'Turbidity', 'Iron',
    'Chlorine', 'Fluoride', 'Pipeline_Age', 'Nitrate', 'Copper',
    'Color', 'Pipeline_Material', 'Lead', 'pH' 
]
# Combine and deduplicate automatically
final_features = list(set(rfe_features + important_features))

# === Step 7.
X_train_final = scaler.fit_transform(X_train_df[final_features])
X_test_final = scaler.transform(X_test_df[final_features])

X_test_final_df = pd.DataFrame(X_test_final, columns=final_features)


In [9]:
# Step 1: Convert to DataFrame for balancing
X_df_balanced = pd.DataFrame(X_train_final, columns=final_features)
y_df_balanced = pd.Series(y_train.values, name="Target")

# Combine into one dataset
balanced_data = pd.concat([X_df_balanced, y_df_balanced], axis=1)

# Step 2: Separate classes
class_0 = balanced_data[balanced_data['Target'] == 0]
class_1 = balanced_data[balanced_data['Target'] == 1]

# Step 3: Balance the dataset (equalize)
n_samples = min(len(class_0), len(class_1), 48000)  # Keep it small for speed
class_0_bal = resample(class_0, replace=False, n_samples=n_samples, random_state=42)
class_1_bal = resample(class_1, replace=False, n_samples=n_samples, random_state=42)

# Combine and shuffle
balanced_train = pd.concat([class_0_bal, class_1_bal]).sample(frac=1, random_state=42)

# Step 4: Split back to X and y
X_balanced = balanced_train.drop(columns=["Target"])
y_balanced = balanced_train["Target"]

In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_recall_curve
import numpy as np

# Step 1: Scale the features
scaler = StandardScaler()
X_balanced_scaled = scaler.fit_transform(X_balanced)
X_test_scaled = scaler.transform(X_test_final_df)

# Step 2: Train Logistic Regression with L1 regularization
lr_model_l1 = LogisticRegression(
    penalty='l1',
    solver='saga',
    max_iter=1000,
    class_weight='balanced',
    random_state=42
)
lr_model_l1.fit(X_balanced_scaled, y_balanced)

# Step 3: Predict probabilities
y_prob = lr_model_l1.predict_proba(X_test_scaled)[:, 1]

# Step 4: Find optimal threshold using precision-recall curve
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
optimal_threshold = thresholds[np.argmax(precision)]
print("Optimal Threshold:", optimal_threshold)


# Optional: Predict with a custom threshold (e.g., 0.3)
custom_thresh = 0.8
y_pred_custom = (y_prob > custom_thresh).astype(int)
print("\nAccuracy (Threshold 0.8):", accuracy_score(y_test, y_pred_custom))
print("Classification Report:\n", classification_report(y_test, y_pred_custom))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_custom))


Optimal Threshold: 0.9999999496059392

Accuracy (Threshold 0.8): 0.8050449419450206
Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.95      0.88    161610
           1       0.65      0.33      0.43     48105

    accuracy                           0.81    209715
   macro avg       0.74      0.64      0.66    209715
weighted avg       0.78      0.81      0.78    209715

Confusion Matrix:
 [[153097   8513]
 [ 32372  15733]]
